# Signal × ETF Screening — Ranked by IS Sortino

Same 70 signal × ETF combinations as research notebook Section 3, but sorted by **In-Sample Sortino** instead of Min OOS Sortino.

This view shows which pairs fitted the IS period (2010–2019) best. Compare against the Min OOS ranking to see overfitting signatures: pairs at the top here but not in the Min OOS ranking fit IS well but failed to generalise.

In [ ]:
import importlib, sys, subprocess

required_packages = ['numpy', 'pandas', 'matplotlib', 'yahooquery', 'yfinance']

missing = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]
if missing:
    print(f'Installing missing packages: {", ".join(missing)} ...')
    for _flags in ([], ['--user'], ['--break-system-packages']):
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing] + _flags)
            break
        except subprocess.CalledProcessError:
            if _flags == ['--break-system-packages']:
                raise
    print('Installation complete.')
else:
    print('All dependencies satisfied.')

try:
    import module
    print('module.py loaded successfully.')
except ModuleNotFoundError:
    raise ImportError(
        'module.py was not found. Place it in the same directory as this notebook.'
    )

import pathlib
import numpy as np
import pandas as pd

importlib.reload(module)

In [ ]:
# Evaluation windows 
IS_START    = '2010-01-01'
IS_END      = '2019-12-31'
OOS1_START  = '2020-01-01'
OOS1_END    = '2025-12-31'
OOS2_START  = '2000-01-01'
OOS2_END    = '2009-12-31'

DATA_DIR = pathlib.Path('data')
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
df_spx = module.load_etf(['^GSPC'], 'spx_ext.csv', OOS2_START, OOS1_END, DATA_DIR)

SECTOR_ETFS = {
    'XLB': 'Materials',    'XLE': 'Energy',       'XLF': 'Financials',
    'XLI': 'Industrials',  'XLK': 'Technology',   'XLP': 'Consumer Staples',
    'XLRE':'Real Estate',  'XLU': 'Utilities',    'XLV': 'Health Care',
    'XLY': 'Consumer Disc.',
}

In [ ]:
etf_ext_csv = DATA_DIR / 'sector_etfs_ext.csv'
if etf_ext_csv.exists():
    df_etfs_ext = pd.read_csv(etf_ext_csv, index_col=0, parse_dates=True)
else:
    dfs_ext = {}
    for tk in SECTOR_ETFS:
        try:
            tmp, _ = module.download_stock_price_data([tk], OOS2_START, OOS1_END)
            dfs_ext[tk] = tmp.iloc[:, 0]
        except Exception as exc:
            print(f'  {tk}: {exc}')
    df_etfs_ext = pd.DataFrame(dfs_ext)
    df_etfs_ext.to_csv(etf_ext_csv)
df_etfs_ext.index = pd.to_datetime(df_etfs_ext.index)

SCREEN_TICKERS = [t for t in SECTOR_ETFS if t in df_etfs_ext.columns]

df_screen_is   = module.slice_period(df_etfs_ext, IS_START,   IS_END)
df_screen_oos1 = module.slice_period(df_etfs_ext, OOS1_START, OOS1_END)
df_screen_oos2 = module.slice_period(df_etfs_ext, OOS2_START, OOS2_END)

SPX_IS_REF   = module.sortino_from_pv(module.slice_period(df_spx, IS_START,   IS_END).iloc[:, 0].to_numpy(dtype=float))
SPX_OOS1_REF = module.sortino_from_pv(module.slice_period(df_spx, OOS1_START, OOS1_END).iloc[:, 0].to_numpy(dtype=float))
SPX_OOS2_REF = module.sortino_from_pv(module.slice_period(df_spx, OOS2_START, OOS2_END).iloc[:, 0].to_numpy(dtype=float))

SIGNAL_GRIDS = {
    'MA Cross': (
        module.ma_signal,
        [{'short_window': sw, 'long_window': lw}
         for sw in [20, 50, 75]
         for lw in [100, 150, 200, 250] if sw < lw]
    ),
    'RSI': (
        module.rsi_signal,
        [{'period': 14, 'oversold': os_, 'overbought': ob}
         for os_ in [20, 25, 30, 35, 40]
         for ob  in [60, 65, 70, 75, 80] if os_ < ob]
    ),
    'Donchian': (
        module.donchian_signal,
        [{'entry_window': ew, 'exit_window': xw}
         for ew in [55, 75, 100, 125, 150, 200]
         for xw in [20, 40, 55, 75, 100] if xw < ew]
    ),
    'MACD': (
        module.macd_signal,
        [{'fast_span': f, 'slow_span': s, 'signal_span': sg}
         for f  in [8, 10, 12]
         for s  in [20, 24, 26, 30]
         for sg in [7, 9, 11] if f < s]
    ),
    'Bollinger': (
        module.bollinger_signal,
        [{'window': w, 'num_std': ns}
         for w  in [10, 20, 30, 50]
         for ns in [1.5, 2.0, 2.5]]
    ),
    'Stochastic': (
        module.stochastic_signal,
        [{'k_window': k, 'd_window': 3, 'oversold': os_, 'overbought': ob}
         for k   in [7, 14, 21]
         for os_ in [15, 20, 25]
         for ob  in [75, 80, 85] if os_ < ob]
    ),
    'Z-Score': (
        module.zscore_signal,
        [{'window': w, 'entry_threshold': et}
         for w  in [10, 20, 40, 60]
         for et in [1.5, 2.0, 2.5]]
    ),
}
SCREEN_SIG_NAMES = list(SIGNAL_GRIDS.keys())

screen_opt = {sn: {} for sn in SCREEN_SIG_NAMES}
for sig_name, (sig_fn, grid) in SIGNAL_GRIDS.items():
    for tk in SCREEN_TICKERS:
        bp, bs = module.screen_optimise(sig_fn, df_screen_is[tk], grid)
        screen_opt[sig_name][tk] = {'params': bp, 'is_sort': bs}

rows = []
for sn in SCREEN_SIG_NAMES:
    for tk in SCREEN_TICKERS:
        is_s = screen_opt[sn][tk]['is_sort']
        mi, _ = module.screen_eval(df_screen_is,   sn, tk, screen_opt, SIGNAL_GRIDS)
        m1, _ = module.screen_eval(df_screen_oos1, sn, tk, screen_opt, SIGNAL_GRIDS)
        m2, _ = module.screen_eval(df_screen_oos2, sn, tk, screen_opt, SIGNAL_GRIDS)
        rows.append({
            'Signal':      sn,
            'ETF':         tk,
            'Sector':      SECTOR_ETFS[tk],
            'IS Sort':     round(is_s,          3),
            'IS CAGR':     round(mi['CAGR'],    4),
            'IS MaxDD':    round(mi['MaxDD'],   4),
            'OOS1 Sort':   round(m1['Sortino'], 3),
            'OOS1 CAGR':   round(m1['CAGR'],   4),
            'OOS1 MaxDD':  round(m1['MaxDD'],  4),
            'OOS1 Sharpe': round(m1['Sharpe'], 3),
            'OOS2 Sort':   round(m2['Sortino'], 3),
            'OOS2 CAGR':   round(m2['CAGR'],   4),
            'OOS2 MaxDD':  round(m2['MaxDD'],  4),
        })

screen_df = pd.DataFrame(rows)
screen_df['Min OOS'] = np.fmin(
    screen_df['OOS1 Sort'].to_numpy(),
    screen_df['OOS2 Sort'].to_numpy()
)
screen_df['Beat OOS1'] = screen_df['OOS1 Sort'] > SPX_OOS1_REF
screen_df['Beat OOS2'] = screen_df['OOS2 Sort'] > SPX_OOS2_REF
screen_df['Beat Both'] = screen_df['Beat OOS1'] & screen_df['Beat OOS2']

# --- Sorted by IS Sortino (descending) ---
ranked = screen_df.sort_values('IS Sort', ascending=False).reset_index(drop=True)
ranked.index += 1  # rank from 1

print(f'ALL {len(ranked)} SIGNAL x ETF COMBINATIONS - ranked by IS Sortino')
print(f'S&P 500:  IS={SPX_IS_REF:.3f}  |  OOS1={SPX_OOS1_REF:.3f}  |  OOS2={SPX_OOS2_REF:.3f}')
print(f'(* = beats S&P 500 in that period)')
print()

hdr = (f'  {"#":>3}  {"Signal":<12} {"ETF":<5} {"Sector":<16}'
       f' {"IS Sort":>8} {"IS CAGR":>8} {"IS MaxDD":>9}'
       f' {"OOS1":>7} {"OOS1 CAGR":>10} {"OOS1 MaxDD":>11}'
       f' {"OOS2":>7} {"OOS2 CAGR":>10} {"OOS2 MaxDD":>11}'
       f' {"Min OOS":>8}')
print(hdr)
print('  ' + '-' * 120)

def fmt_pct(v):
    return f'{v:.2%}' if v == v else '   nan'

for rank, r in ranked.iterrows():
    f1 = '*' if r['Beat OOS1'] else ' '
    f2 = '*' if r['Beat OOS2'] else ' '
    fb = '<<' if r['Beat Both'] else '  '
    print(f'  {rank:>3}  {r["Signal"]:<12} {r["ETF"]:<5} {r["Sector"]:<16}'
          f' {r["IS Sort"]:>8.3f} {fmt_pct(r["IS CAGR"]):>8} {fmt_pct(r["IS MaxDD"]):>9}'
          f' {r["OOS1 Sort"]:>6.3f}{f1} {fmt_pct(r["OOS1 CAGR"]):>10} {fmt_pct(r["OOS1 MaxDD"]):>11}'
          f' {r["OOS2 Sort"]:>6.3f}{f2} {fmt_pct(r["OOS2 CAGR"]):>10} {fmt_pct(r["OOS2 MaxDD"]):>11}'
          f' {r["Min OOS"]:>8.3f}{fb}')

print()
print('<< = beats S&P 500 in BOTH OOS periods')
n_both = int(screen_df['Beat Both'].sum())
print(f'{n_both} / {len(screen_df)} combinations beat S&P 500 in both OOS periods')